# Model Serving Architecture

## Model Serving Architecture

#### **High-Level Overview of Infrastructure Options**
1. **On-Premises**:
   - Full control over hardware and software infrastructure.
   - Flexibility to adapt quickly to changes.
   - Costly: Requires procurement, installation, configuration, and maintenance.
   - Often used by larger companies due to economies of scale and in-house expertise.

2. **Cloud Providers**:
   - Outsource infrastructure needs to cloud vendors like Amazon (AWS), Google Cloud Platform (GCP), and Microsoft Azure.
   - Vendor-managed hardware, scaling, and monitoring.
   - Offers pipeline management services and software tools (e.g., AutoML).
   - Preferred by smaller companies due to easier management and scaling options.

#### **Model Serving**
- **Model Serving** refers to the deployment of machine learning models where they are made available to clients for inference.
- Models are typically saved to the file system and managed by the model server.
- **Key Functions of a Model Server**:
  - Loads and instantiates the model.
  - Manages multiple model versions (useful for A/B testing or different user groups).
  - Exposes APIs (e.g., REST or RPC) for clients to make predictions.

#### **Infrastructure Impact on Model Serving**
- **On-Premises**: 
  - Choose and configure the model server (e.g., TensorFlow Serving, Kubeflow, NVIDIA Triton).
- **Cloud**:
  - Create virtual machines for full control over model serving.
  - Use pre-built cloud tools and services (e.g., AutoML on GCP, SageMaker Autopilot on AWS).

#### **Popular Model Servers**
1. **TensorFlow Serving**:
   - Designed for serving TensorFlow models in production environments.
   - Supports multiple models and versions.
   - Provides APIs for efficient inference requests (REST, gRPC).

2. **TorchServe**:
   - Tailored for PyTorch models.
   - Provides multi-model serving and model versioning.
   - Supports REST and gRPC interfaces for making predictions.

3. **Kubeflow Serving**:
   - Part of the Kubeflow ecosystem, optimized for Kubernetes environments.
   - Supports various ML frameworks (e.g., TensorFlow, PyTorch).
   - Enables scalable, production-ready model serving in containerized setups.

4. **NVIDIA Triton Inference Server**:
   - Optimized for GPU-based inference.
   - Supports multiple frameworks (e.g., TensorFlow, PyTorch, ONNX).
   - Provides both HTTP/REST and gRPC endpoints.
   - Includes features for batching, model version management, and performance monitoring.

---

#### **Key Concepts to Remember**
- **On-Premises vs. Cloud**: Trade-offs between control and convenience; large companies favor on-premises, while smaller ones lean towards cloud.
- **Model Server**: A critical component that loads models, manages versions, handles input formatting, and exposes APIs for inference.
- **Popular Servers**: TensorFlow Serving, TorchServe, Kubeflow Serving, NVIDIA Triton Inference Server—each with different strengths depending on the environment and framework.

---

**Short Notes for Quick Revision**:
- **On-Premises**: Full control, higher cost, flexibility.
- **Cloud**: Outsourced, managed, scalable, easier for smaller teams.
- **Model Server**: Manages model loading, versioning, and API exposure.
- **TensorFlow Serving**: Best for TensorFlow models, efficient API.
- **TorchServe**: For PyTorch, multi-model, REST/gRPC support.
- **Kubeflow Serving**: Kubernetes-focused, multi-framework support.
- **NVIDIA Triton**: GPU-optimized, multi-framework, powerful batching and performance features.

## Model Servers: TensorFlow Serving

#### **Overview of TensorFlow Serving**
- **TensorFlow Serving** is a flexible, high-performance serving system designed primarily for TensorFlow models but extendable to other types.
- Supports both **batch** and **real-time inference**:
  - **Batch inference**: Ideal for tasks like recommendation engines with high prediction volumes.
  - **Real-time inference**: Suitable for tasks like image classification requiring quick responses.
- **Multi-model serving**: Supports multiple models for the same task, useful for A/B testing and audience segmentation.
- Offers **RPC and REST APIs** for clients to interact with the model server.

---

#### **Key Architectural Components**
1. **Servable**:
   - Central abstraction in TensorFlow Serving.
   - Represents the underlying object clients use for computation (e.g., inference or lookups).
   - A typical servable is a TensorFlow SavedModel but can also include other objects like lookup tables or embeddings.
   - Provides flexibility by allowing different types and interfaces.

2. **Loader**:
   - Manages the lifecycle of a servable.
   - Provides standardized APIs for loading and unloading servables, independent of specific learning algorithms or data.
   - Creates and manages multiple versions of servables, known as **aspired versions**.

3. **Source**:
   - Communicates the set of aspired versions for a servable stream to the manager.
   - When a new version of the model is detected, it updates the manager with this information.
   - Ensures that the servable versions are updated dynamically as new models or weights become available.

4. **Dynamic Manager**:
   - Handles the full lifecycle of servables: loading, serving, and unloading.
   - Listens to sources and tracks the versions of servables.
   - Uses a **version policy** to decide which versions of the servable to load and unload.
   - Ensures there is enough memory for new servables and coordinates their instantiation.

5. **Servable Handle**:
   - Provides an external interface for clients to interact with the model.
   - Clients request handles to servables, and the dynamic manager returns the appropriate version for inference.

---

#### **Example Workflow**
1. **Model Update**:
   - A source represents a TensorFlow graph with frequently updated model weights stored on disk.
   - When new model weights are available, the source detects the update and creates a loader containing a pointer to the updated model data.

2. **Dynamic Manager's Role**:
   - The source notifies the dynamic manager of the new aspired version.
   - The dynamic manager checks memory availability and applies the version policy.
   - If appropriate, the manager instructs the loader to instantiate the TensorFlow graph as a new servable with the updated weights.

3. **Inference**:
   - A client requests a handle to the latest version of the model.
   - The dynamic manager returns a handle to the new servable version.
   - The client can then run inference using the updated servable.

---

**Short Notes for Quick Revision**:
- **Servable**: Core computation object (e.g., SavedModel, lookup table).
- **Loader**: Manages servable lifecycle, standardizes loading/unloading APIs.
- **Source**: Communicates aspired versions, detects new model versions.
- **Dynamic Manager**: Manages servable lifecycle, applies version policy, tracks memory.
- **Servable Handle**: External client interface for accessing models.
- **Batch vs. Real-Time Inference**: Batch for bulk predictions (e.g., recommendations); Real-time for quick responses (e.g., image classification).


## Model Servers: Other Providers

#### **Triton Inference Server (NVIDIA) Overview**
- **Triton Inference Server** is an open-source inference serving software that enables scalable AI model deployment across various frameworks, including TensorFlow, TensorRT, PyTorch, ONNX Runtime, or custom frameworks.
- **Deployment Options**:
  - Deploy models from local storage or cloud platforms (e.g., Google Cloud, AWS).
  - Supports deployment on GPU or CPU-based infrastructures in the cloud, data center, or edge.
  
- **Concurrency and Utilization**:
  - Runs multiple models concurrently on a single GPU using CUDA streams.
  - On multi-GPU servers, Triton automatically creates an instance of each model on each GPU.
  - Optimizes GPU utilization without extra user coding.

- **Inference Types**:
  - Supports **low-latency real-time inference** and **batch inference** to maximize GPU and CPU utilization.
  - Provides built-in support for **streaming inference** and **shared memory** to improve performance.
  - Models can use either system memory or **CUDA shared memory**, reducing HTTP or gRPC overhead and increasing overall efficiency.

- **Model Ensemble**:
  - Supports **model ensemble**, enabling complex model pipelines by chaining models together.

- **Integration and Scalability**:
  - Integrates with **Kubernetes** for orchestration, metrics, and auto-scaling.
  - Works with **Kubeflow** and **Kubeflow Pipelines** for end-to-end AI workflows.
  - Exports **Prometheus metrics** for monitoring (e.g., GPU utilization, latency, memory usage, inference throughput).
  - Supports scaling across multiple servers to handle increasing inference loads.
  
- **Model Control API**:
  - Manages tens or hundreds of models, allowing dynamic loading/unloading based on model control configuration.
  - Supports heterogeneous clusters with both GPUs and CPUs, standardizing inference across platforms.
  - Dynamically scales out to handle peak loads using both CPUs and GPUs.

---

#### **TorchServe (AWS & Facebook) Overview**
- **TorchServe** is a model serving framework for PyTorch models, co-developed by AWS and Facebook.
- **Deployment Features**:
  - Eliminates the need to manually build custom model-serving solutions for PyTorch models.
  - Supports deploying models in both **eager mode** and **graph mode**.
  
- **Concurrency and Model Management**:
  - Supports serving multiple models simultaneously.
  - Enables **A/B testing** through model versioning and dynamic model loading/unloading.
  
- **Extensibility and Monitoring**:
  - Open-source and extensible to fit deployment needs.
  - Provides detailed logs, customizable metrics, and plugins for common tasks (e.g., logs, snapshots, reporting).
  
- **Architecture**:
  - **Front-end**: Handles client requests, responses, and model lifecycle management.
  - **Back-end**: Uses **model workers** running instances of models loaded from a **model store** to perform inference.
  - Supports running multiple workers simultaneously, enabling higher throughput by handling more requests concurrently.
  
- **Storage and API Support**:
  - Models can be loaded from either cloud storage or local hosts.
  - Supports APIs for **management and inference**.

---

#### **Kubeflow Serving Overview**
- **Kubeflow Serving** integrates with Kubernetes to provide **serverless inference** through abstraction.
- Supports multiple frameworks, including TensorFlow and PyTorch.
- Provides scalability and orchestration features through Kubernetes, with more information available in the additional readings.

---

### **Scaling Applications**
- **Horizontal Scaling**: Increasing the number of instances (e.g., servers) to distribute the load across them. This is essential for scaling out in distributed systems.
- **Vertical Scaling**: Enhancing the capacity of a single instance (e.g., adding more CPU or GPU resources) to handle more load.
- **Virtualization & Containers**: Use virtualization or container technologies (e.g., Docker) to efficiently manage resources and isolate workloads.
- **Container Orchestration**: Tools like **Kubernetes** manage containerized applications, enabling auto-scaling, fault tolerance, and streamlined deployments.

These foundational scaling strategies are critical for serving applications at the appropriate scale in production environments.

# Scaling Infrastructure

#### **Understanding Scaling**

Scaling is crucial for managing the efficiency and cost-effectiveness of training and serving machine learning models. Here’s a breakdown:

**1. **Vertical Scaling:**
- **Concept**: Involves using more powerful hardware (e.g., upgrading CPUs, adding RAM, using newer GPUs).
- **Example**: Upgrading from a car that holds 5 people to one that holds 10 people to move more people faster.
- **Pros**: Simple to implement; does not require managing multiple instances.
- **Cons**: Limited by the maximum capacity of a single machine; often requires downtime for upgrades.

**2. **Horizontal Scaling:**
- **Concept**: Adding more machines to the network to handle increased load (e.g., adding more GPUs or CPUs).
- **Example**: Using 20 cars, each holding 5 people, to transport 100 people simultaneously.
- **Pros**: Elastic and cost-effective; you can scale up and down based on demand, paying only for what you use.
- **Cons**: Requires management of multiple instances; coordination between instances can be complex.

**Key Points**:
- **Elasticity**: Horizontal scaling allows dynamic adjustment of resources based on demand, without needing to take applications offline.
- **Cost Efficiency**: You can lease resources as needed, reducing overhead compared to maintaining a single large instance.

---

#### **Containers for Horizontal Scaling**

**Containers** offer a lightweight alternative to virtual machines, enhancing horizontal scaling:

**1. **Traditional VM Architecture:**
- **Concept**: Runs multiple instances of applications on separate virtual machines, each with its own OS.
- **Pros**: Provides isolation and full OS functionality.
- **Cons**: More resource-intensive; each VM requires a full OS, which can be inefficient.

**2. **Container Architecture:**
- **Concept**: Runs applications in isolated environments (containers) without needing a separate OS for each instance.
- **Pros**:
  - **Lightweight**: Containers share the host OS kernel, allowing for more efficient use of resources.
  - **Flexibility**: Containers can run on any hardware supporting container runtimes, simplifying deployment.
  - **Scalability**: Easier to scale horizontally due to reduced resource overhead.

**3. **Docker**:
- **Concept**: Popular container runtime that started with Linux but now supports multiple OSes.
- **Pros**: Facilitates deployment, especially for complex applications with dependencies.
- **Cons**: Requires management of container instances and orchestration.

---

#### **Container Orchestration**

**Container orchestration** manages the lifecycle of containers, including deployment, scaling, and maintenance:

**1. **Kubernetes**:
- **Concept**: Open-source system for automating deployment, scaling, and management of containerized applications.
- **Features**:
  - **Logical Units**: Groups containers into units for easier management.
  - **Scalability**: Automates scaling based on demand.
  - **Service Management**: Manages service availability and load balancing.
- **Use Case**: Example includes the Financial Times using Kubernetes to scale microservices.

**2. **Docker Swarm**:
- **Concept**: Native clustering and scheduling tool for Docker containers.
- **Features**:
  - **Simple Deployment**: Easier to set up compared to Kubernetes for smaller scale use cases.
  - **Scaling**: Manages container scaling and distribution.

**3. **Kubeflow**:
- **Concept**: Built on Kubernetes, designed for machine learning workflows.
- **Features**:
  - **End-to-End ML**: Handles data ingestion, training, model management, and deployment.
  - **Portability**: Can run on any environment where Kubernetes is supported.
- **Use Case**: Facilitates scalable ML workflows, integrating seamlessly with Kubernetes.

---

### **Practical Application**

For efficient management of machine learning workloads:
- **Horizontal Scaling**: Leverage cloud platforms for dynamic scaling and cost efficiency.
- **Containers**: Use Docker for simplified deployment and management of dependencies.
- **Orchestration**: Implement Kubernetes or Docker Swarm for scaling and managing containerized applications. Explore **Kubeflow** for comprehensive ML workflow management.

### **Next Steps**
- **Experiment with Kubernetes** and **Kubeflow** for ML workflows to understand their integration and scaling capabilities.
- **Explore Docker** and **container orchestration tools** to manage and scale your applications effectively. 

Feel free to dive into the additional readings and practical labs to get hands-on experience with these tools.

# Online Inference

Here’s a summary of the key points covered about optimizing online inference:

### **Key Concepts for Optimizing Online Inference**

1. **Optimization Areas for Inference**:
   - **Infrastructure**: Ensure you have scalable hardware and consider using containerized or virtualized environments. 
   - **Model Architecture**: Balance between inference speed and accuracy. Sometimes a slightly less accurate model might be more cost-effective if it performs faster.
   - **Model Compilation**: Tailor the model to the specific hardware for deployment. Post-training optimization can reduce memory usage and latency.

2. **Latency**:
   - Optimize the end-to-end latency from user action to response.
   - Latency can be influenced by various factors including data transport, inference execution, and rendering of results.
   - Aim to minimize latency across the entire application, not just the model inference part.

3. **Throughput**:
   - Measure and optimize the number of requests handled per unit time.
   - This is particularly critical for high-traffic scenarios, though it’s also important for non-customer facing systems like data processing.

4. **Cost**:
   - Consider not just hardware costs but also engineering, testing, software licenses, and opportunity costs.
   - Optimize both latency and throughput to manage costs effectively.

5. **Application Layer Optimizations**:
   - **Caching**: Use faster data storage solutions for frequently accessed data. For example, use in-memory caching systems like Amazon’s DynamoDB or Google Cloud’s Memorystore.
   - **Trade-Offs**: Balance the cost of fast storage against the benefit of reduced latency. Not all data may need to be cached; optimize based on usage patterns.

6. **Data Handling**:
   - Optimize preprocessing and post-processing of data to enhance performance. Ensure that data passed to and from the model is efficiently managed.

### **Strategies for Optimization**

- **Infrastructure Scaling**: Use horizontal scaling (adding more instances) or vertical scaling (upgrading existing hardware) based on needs.
- **Model Efficiency**: Adjust the model architecture and compile it specifically for the target hardware.
- **Application Optimization**: Implement caching for high-demand data and consider application-level changes to reduce latency.

### **Technologies Mentioned**

- **Containerization**: Docker for lightweight deployment.
- **Container Orchestration**: Kubernetes for managing containerized applications and scaling.
- **In-Memory Caching**: Tools like DynamoDB, Memorystore, and Bigtable for fast data retrieval.

By focusing on these aspects, you can improve the performance and efficiency of your online inference systems.

# Data Preprocessing

Here’s a summary of the key points related to preprocessing and post-processing data for online inference:

### **Preprocessing**

1. **Data Conversion**: Data coming into the system may need to be converted into the format that the model expects. For example, a language model may require sentences to be transformed into high-dimensional vectors.

2. **Data Cleansing**: This involves correcting or removing invalid values from incoming data. For instance, resizing an image that’s too large before processing.

3. **Feature Tuning**: Adjusting data to fit model requirements. For example:
   - **Normalization**: Converting pixel values from 0-255 to 0-1 for image processing.
   - **Encoding**: Converting text into vocabulary representations.

4. **Feature Construction**: Creating new features from existing data. For example:
   - **Feature Crossing**: Combining features like the number of rooms and size to get total floor space.
   - **Polynomial Expansion**: Adding new features based on formulas.

5. **Representation Transformation**: Transforming data representations to fit model needs, such as one-hot encoding.

6. **Smart Caching**: Pre-transforming and caching frequently used features to reduce processing time.

### **Post-Processing**

1. **Data Conversion**: After the model generates predictions, you may need to convert them back into a user-friendly format. For instance, converting vectors into readable text.

2. **Reverse Operations**: Similar to preprocessing but in reverse, post-processing involves tasks like converting model outputs into meaningful results for the user.

### **Tools for Preprocessing**

- **Apache Beam**: A unified model for batch and streaming data processing.
- **TensorFlow Transform (TFT)**: A library for preprocessing data for TensorFlow models.

These tools help automate and optimize the preprocessing workflow, making it easier to manage complex transformations and large datasets.

# Batch Inference Scenarios

Here’s a quick summary of the key points:

### **Batch Inference Overview**

**Definition**:
- Batch inference involves generating predictions for a large number of data points at once, rather than in real-time. It’s useful for scenarios where immediate predictions aren’t required.

**Advantages**:
1. **Complex Models**: You can use more complex models without worrying about inference time constraints.
2. **Caching**: Predictions are typically stored for future use, reducing the need for real-time caching.
3. **Data Retrieval**: Can wait for data retrieval, which is more efficient without real-time constraints.

**Disadvantages**:
1. **Update Latency**: Predictions can be delayed, potentially resulting in outdated predictions.
2. **Real-Time Limitations**: Not suitable for scenarios requiring immediate responses.

**Metrics**:
- **Throughput**: Focus on increasing the number of predictions processed per unit time. Latency is less critical compared to real-time systems.

### **Use Cases**

1. **E-Commerce Recommendations**:
   - Batch predictions can be used to generate product recommendations based on historical interactions. Predictions are stored and used to personalize the user experience without real-time constraints.

2. **Sentiment Analysis**:
   - Reviews can be analyzed periodically (e.g., weekly) to gauge customer sentiment. This allows the use of complex models like LSTM for improved accuracy.

3. **Demand Forecasting**:
   - Time series models (e.g., ARIMA, SARIMA, RNN) can forecast future demand for inventory optimization. Batch predictions are ideal as they don't require immediate results.

### **Optimization Strategies**

- **Hardware Accelerators**: Use GPUs or TPUs to handle large volumes of data efficiently.
- **Scalability**: Increase the number of servers or workers to handle more data simultaneously.

### **Key Considerations**

- **Latency vs. Throughput**: While throughput is a priority, understanding the balance between latency and processing speed is essential, especially in batch settings.

This summary should give you a solid foundation in batch inference, its advantages, limitations, and practical applications. If you have specific questions or need more details on any aspect, feel free to ask!

# Batch Processing with ETL

When dealing with different types of data and their processing needs, it's important to understand the various frameworks and approaches that can be used. Here's a summary based on the scenario you've described:

### ETL Pipelines

**ETL** stands for Extract, Transform, Load, and is crucial for preparing data for machine learning predictions. Here's a breakdown of the ETL process:

1. **Extract**: Data is gathered from various sources such as CSV files, log files, APIs, or streaming sources.
2. **Transform**: Data is processed to fit the requirements of the ML models. This might involve cleaning, normalization, feature engineering, etc.
3. **Load**: Transformed data is stored in a destination like a data warehouse or a data lake for further use.

### Data Processing Frameworks

**Batch Processing Frameworks**:
- **Apache Spark**: A powerful distributed computing system for batch processing. It can handle large volumes of data and perform complex transformations efficiently.
- **Google Cloud Dataflow**: A managed service that can handle batch and stream processing using Apache Beam.

**Streaming Data Frameworks**:
- **Apache Kafka**: A distributed streaming platform that can handle real-time data feeds.
- **Google Cloud Pub/Sub**: A messaging service for building event-driven systems and real-time analytics.

### Handling Different Data Types

- **Batch Data**: Typically stored in data lakes or warehouses, processed in bulk at scheduled intervals.
- **Streaming Data**: Arrives in real-time and needs to be processed immediately to keep up with the data flow.

### Example Scenario

On Google Cloud Platform (GCP):
- **Data Sources**: CSV files, JSON, APIs, data lakes.
- **ETL Processing**: Use Apache Beam with Google Cloud Dataflow for both batch and streaming data.
- **Data Warehousing**: Store transformed data in BigQuery or a similar data warehouse.
- **Streaming Data Handling**: Use Apache Kafka or Google Cloud Pub/Sub to manage and process real-time data streams.

In a practical example, you might use Google Cloud Dataflow to handle a batch processing job that processes molecular data for ML purposes, giving you a hands-on understanding of how ETL frameworks work in practice.